In [6]:
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnableSequence,RunnableParallel,RunnablePassthrough,RunnableLambda
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from youtube_transcript_api import YouTubeTranscriptApi

In [2]:
load_dotenv()

True

# Indexing

# Step1 : Document Loader

In [15]:
video_id = "Gfr50f6ZBvo"

try:
    transcript_list = YouTubeTranscriptApi().fetch(video_id, languages=["en"])

except Exception as e:
    print(f"No transcript found: {e}")

In [22]:
transcript_lt = []

for i in range(len(transcript_list)):
    transcript_lt.append(transcript_list[i].text) 

In [25]:
transcript = " ".join(transcript_lt)

In [27]:
type(transcript)

str

# Step2 : Text Splitter

In [28]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000 , chunk_overlap = 200)
chunks = splitter.create_documents([transcript])

In [30]:
type(chunks)

list

In [29]:
len(chunks)

168

In [31]:
chunks[0]

Document(metadata={}, page_content="the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to inter

# Step3 : Vector Store

In [ ]:
vector_store = FAISS.from_documents(
    documents = chunks,
    embedding = OpenAIEmbeddings(model = "text-embedding-3-small")
)

In [37]:
vector_store.index_to_docstore_id

{0: '3eaa4bb6-f80b-403d-a1f8-ba6840c84108',
 1: '770b41fa-4fce-43ff-890d-cce8e022e075',
 2: 'ea14b101-cd41-40c5-92e6-aa330fce63ef',
 3: '454d1639-7a16-4a0d-8c82-d2cd9657f419',
 4: '44438f1e-317e-402e-9e2a-2de1e008b413',
 5: 'cb0355c0-c335-43b2-8a2f-55529ae4b6c3',
 6: '528dc91a-2f61-4b20-a3fd-7d85ae962385',
 7: 'cf1bb0a8-1393-43e3-aaa9-23ed9f1945ac',
 8: '1e206996-e138-4619-8e34-f05a322e6365',
 9: '6ebffb6e-7988-4ea0-bf95-e606cfd612b1',
 10: 'af8ae513-384a-442d-bcb3-2bc922c8e6d3',
 11: 'deda3c01-8352-4abe-926e-f4fdfe352d9e',
 12: 'c51aeb53-a176-4e2f-85d8-1ec8b56f64e6',
 13: '557a8e3a-d3d4-41b0-bf78-62af388acbc2',
 14: '40dc4eac-aab9-4e97-a1d2-401ca8d52937',
 15: 'b73d3e63-fe89-4315-966f-f739f135dc25',
 16: '03ef0483-b194-4266-b43e-3176b5d9cac2',
 17: 'f476a3c1-5567-4d0a-8f72-f1154658fffc',
 18: '3dd69280-d3fa-4613-b352-f463831e9ff1',
 19: '28892c2d-6f82-4785-ae7f-04746eb8d7d7',
 20: '1a617579-1d1a-40bf-82a9-f6999a30d426',
 21: '05443ab2-b114-401a-b10d-6a498f32efb7',
 22: 'b50640ec-b36f-

# Step4 : Retriever

In [38]:
retriever = vector_store.as_retriever(search_kwargs = {"k":4})

# Retrieval

In [40]:
relevant_docs = retriever.invoke('what is deepmind')

In [42]:
len(relevant_docs)

4

In [45]:
relevant_docs[0].page_content

"the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get good enough

In [48]:
content = " ".join([relevant_docs[i].page_content for i in range(len(relevant_docs))])

In [49]:
content

"the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get good enough

# Augumentation

-> Combine prompt with relevant documents

In [66]:
prompt = PromptTemplate(
    template = '''
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    ''',
    input_variables = ["context" , "question"]
)

In [52]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [53]:
retrieved_docs

[Document(id='15325410-bada-4506-b40e-26ff539b6951', metadata={}, page_content="so we with this problem and we published it in a nature paper last year uh we held the fusion that we held the plasma in specific shapes so actually it's almost like carving the plasma into different shapes and control and hold it there for the record amount of time so um so that's one of the problems of of fusion sort of um solved so i have a controller that's able to no matter the shape uh contain it continue yeah contain it and hold it in structure and there's different shapes that are better for for the energy productions called droplets and and and so on so um so that was huge and now we're looking we're talking to lots of fusion startups to see what's the next problem we can tackle uh in the fusion area so another fascinating place in a paper title pushing the frontiers of density functionals by solving the fractional electron problem so you're taking on modeling and simulating the quantum mechanical 

In [58]:
context_text = " ".join([retrieved_docs[i].page_content for i in range(len(retrieved_docs))])

In [59]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

# Generation

In [60]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

In [62]:
response = llm.invoke(final_prompt)

In [63]:
response.content

'Yes, the topic of nuclear fusion is discussed in the video. The discussion includes the following points:\n\n1. The speaker mentions a problem they published in a Nature paper regarding holding plasma in specific shapes for a record amount of time, which is significant for fusion energy production.\n2. They describe their controller that can contain and hold plasma in various shapes, with some shapes being better for energy production.\n3. The speaker talks about collaborating with EPFL in Switzerland, which has a test reactor, to explore bottleneck problems in fusion and how AI methods can address these challenges.\n4. They emphasize the importance of energy and climate as areas where AI can help, alongside disease and biology.\n5. The speaker mentions the challenges in fusion, including physics, material science, and engineering, and the need for collaboration with domain experts.\n6. They reference their work on using deep reinforcement learning for the magnetic control of tokamak 

# Chaining the entire chatbot process

In [67]:
parser = StrOutputParser()

In [64]:
def context_text(retrieved_docs):
    return " ".join([retrieved_docs[i].page_content for i in range(len(retrieved_docs))])

In [72]:
parallel_chain = RunnableParallel({
"context" : retriever |  RunnableLambda(context_text),
"question" : RunnablePassthrough()
})

In [73]:
output_chain = prompt | llm | parser

In [74]:
final_chain = parallel_chain | output_chain

In [77]:
question = "Summarize the video"

In [76]:
final_chain.invoke(question)

'Yes, the video discusses the possibility of alien civilizations, the search for them, and various theories about why we may not have encountered them yet. It explores ideas about communication, consciousness, and the likelihood of being alone in the universe.'